In [1]:
!pip install xgboost lightgbm torch matplotlib scikit-learn pandas numpy

In [2]:
# ================================
# FULL LINK PREDICTION EXPERIMENT NOTEBOOK
# ================================

import numpy as np
import pandas as pd
import random, os, json
import matplotlib.pyplot as plt
from datetime import datetime
from collections import defaultdict

from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import torch
import torch.nn as nn
import torch.optim as optim

from matplotlib.backends.backend_pdf import PdfPages


# ================================
# CONFIG
# ================================
class Config:
    DATA_DIR = "Data"
    OUTPUT_DIR = "results"
    RANDOM_STATE = 475
    N_SPLITS = 5

    CLUSTER_RANGE = range(5, 26)

    GBDT_PARAMS = dict(n_estimators=12, max_depth=5, min_samples_leaf=13)

    SVM_GRID = {
        "C": [0.1, 1, 10],
        "gamma": [0.1, 0.01]
    }

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)


# ================================
# DATA LOADER
# ================================
class DataLoader:
    def load(self):
        self.disease_sim = pd.read_csv(f"{cfg.DATA_DIR}/disease_similarity.csv", header=None).values
        self.snorna_sim  = pd.read_csv(f"{cfg.DATA_DIR}/snoRNA_similarity.csv", header=None).values
        self.adjacency   = pd.read_csv(f"{cfg.DATA_DIR}/known_snoRNA_disease.csv", header=None).values
        return self


# ================================
# PAIR BUILDER
# ================================
class PairBuilder:
    def __init__(self, D, S, A):
        self.D = D
        self.S = S
        self.A = A

    def split_known_unknown(self):
        known, unknown = [], []
        for r in range(self.A.shape[0]):
            for d in range(self.A.shape[1]):
                (known if self.A[r,d]==1 else unknown).append((r,d))
        return known, unknown

    def feature(self, r, d):
        return np.concatenate([self.D[d], self.S[r]])


# ================================
# CLUSTER SAMPLER
# ================================
class ClusterSampler:
    def __init__(self, k):
        self.k = k

    def fit(self, feats):
        self.labels = KMeans(self.k, random_state=cfg.RANDOM_STATE).fit_predict(feats)
        return self.labels

    def balanced_sample(self, unknown, labels, n_known):
        buckets = defaultdict(list)
        for p,l in zip(unknown, labels):
            buckets[l].append(p)

        sampled = []
        for b in buckets:
            k = int(len(buckets[b]) / len(labels) * n_known)
            sampled.extend(random.sample(buckets[b], min(k, len(buckets[b]))))

        return sampled


# ================================
# DATASET BUILDER
# ================================
class DatasetBuilder:
    def __init__(self, pb):
        self.pb = pb

    def build(self, neg, pos):
        X, y = [], []
        for r,d in neg:
            X.append(self.pb.feature(r,d)); y.append(0)
        for r,d in pos:
            X.append(self.pb.feature(r,d)); y.append(1)
        return np.array(X), np.array(y)


# ================================
# SANITY CHECK
# ================================
class SanityChecker:
    @staticmethod
    def check(X,y):
        assert len(X)==len(y)
        assert not np.isnan(X).any()
        assert set(np.unique(y))=={0,1}
        print("✔ Dataset OK", X.shape)


# ================================
# BASE CLASSIFIER
# ================================
class BaseClassifier:
    def fit(self,X,y): pass
    def predict_proba(self,X): pass


# ================================
# MODELS
# ================================
class SklearnClassifier(BaseClassifier):
    def __init__(self, model):
        self.model=model
        self.scaler=StandardScaler()
    def fit(self,X,y):
        self.model.fit(self.scaler.fit_transform(X),y)
    def predict_proba(self,X):
        return self.model.predict_proba(self.scaler.transform(X))[:,1]


class GBDTSVMClassifier(BaseClassifier):
    def __init__(self):
        self.gbdt = GradientBoostingClassifier(**cfg.GBDT_PARAMS)
        self.ohe = OneHotEncoder()
        self.svm = GridSearchCV(SVC(probability=True), cfg.SVM_GRID, cv=3)

    def fit(self,X,y):
        self.gbdt.fit(X,y)
        leaves=self.gbdt.apply(X)[:,:,0]
        self.ohe.fit(leaves)
        self.svm.fit(self.ohe.transform(leaves),y)

    def predict_proba(self,X):
        leaves=self.gbdt.apply(X)[:,:,0]
        return self.svm.predict_proba(self.ohe.transform(leaves))[:,1]


class XGBClassifierWrapper(BaseClassifier):
    def __init__(self):
        self.model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                    subsample=0.8, colsample_bytree=0.8,
                                    eval_metric="auc", random_state=cfg.RANDOM_STATE)

    def fit(self,X,y):
        self.model.fit(X,y)

    def predict_proba(self,X):
        return self.model.predict_proba(X)[:,1]


class LGBMClassifierWrapper(BaseClassifier):
    def __init__(self):
        self.model = LGBMClassifier(n_estimators=400, learning_rate=0.03,
                                    subsample=0.8, colsample_bytree=0.8,
                                    random_state=cfg.RANDOM_STATE)

    def fit(self,X,y):
        self.model.fit(X,y)

    def predict_proba(self,X):
        return self.model.predict_proba(X)[:,1]


# ================================
# DNN
# ================================
class DNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim,512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256,1), nn.Sigmoid()
        )
    def forward(self,x): return self.net(x)

class DNNClassifier(BaseClassifier):
    def __init__(self,input_dim):
        self.model=DNN(input_dim)
        self.scaler=StandardScaler()
        self.opt=optim.Adam(self.model.parameters(), lr=1e-3)
        self.loss=nn.BCELoss()

    def fit(self,X,y,epochs=30):
        X=self.scaler.fit_transform(X)
        X=torch.tensor(X,dtype=torch.float32)
        y=torch.tensor(y,dtype=torch.float32).view(-1,1)

        for e in range(epochs):
            pred=self.model(X)
            l=self.loss(pred,y)
            self.opt.zero_grad()
            l.backward()
            self.opt.step()

    def predict_proba(self,X):
        X=self.scaler.transform(X)
        X=torch.tensor(X,dtype=torch.float32)
        return self.model(X).detach().numpy().flatten()


# ================================
# ENSEMBLE
# ================================
class EnsembleClassifier(BaseClassifier):
    def __init__(self, models):
        self.models=models

    def fit(self,X,y):
        for m in self.models: m.fit(X,y)

    def predict_proba(self,X):
        return np.mean([m.predict_proba(X) for m in self.models], axis=0)


# ================================
# CROSS VALIDATOR
# ================================
class CrossValidator:
    def __init__(self, clf):
        self.clf=clf
        self.skf=StratifiedKFold(cfg.N_SPLITS, shuffle=True, random_state=cfg.RANDOM_STATE)

    def run(self,X,y):
        scores=[]
        for tr,te in self.skf.split(X,y):
            self.clf.fit(X[tr],y[tr])
            scores.append(roc_auc_score(y[te], self.clf.predict_proba(X[te])))
        return np.mean(scores), np.std(scores)


# ================================
# MODEL FACTORY
# ================================
def get_models(input_dim):
    return {
        "GBDT-SVM": GBDTSVMClassifier(),
        "XGB": XGBClassifierWrapper(),
        "LGB": LGBMClassifierWrapper(),
        "RF": SklearnClassifier(RandomForestClassifier(n_estimators=300)),
        "DNN": DNNClassifier(input_dim),
        "ENSEMBLE": EnsembleClassifier([
            XGBClassifierWrapper(),
            LGBMClassifierWrapper()
        ])
    }


# ================================
# LOAD DATA
# ================================
data = DataLoader().load()
pb = PairBuilder(data.disease_sim, data.snorna_sim, data.adjacency)

known, unknown = pb.split_known_unknown()

unknown_feats = [pb.feature(r,d) for r,d in unknown]

results_summary = []
models_final = ["GBDT-SVM","XGB","LGB","RF","DNN","ENSEMBLE"]

# ================================
# PDF REPORT
# ================================
pdf_path = f"{cfg.OUTPUT_DIR}/ML_Report.pdf"
COMPARISON_CSV = f"{cfg.OUTPUT_DIR}/comparison.csv"

with PdfPages(pdf_path) as pdf_pages:

    for k in cfg.CLUSTER_RANGE:
        print("\n===== CLUSTERS:", k, "=====")

        sampler = ClusterSampler(k)
        labels = sampler.fit(unknown_feats)

        sampled_unknowns = sampler.balanced_sample(unknown, labels, len(known))
        X,y = DatasetBuilder(pb).build(sampled_unknowns, known)

        SanityChecker.check(X,y)

        models = get_models(X.shape[1])

        for name, clf in models.items():
            cv = CrossValidator(clf)
            mean_auc, std_auc = cv.run(X,y)

            results_summary.append({
                "clusters": k,
                "model": name,
                "mean_roc_auc": mean_auc,
                "std": std_auc
            })

            print(f"{name}: {mean_auc:.4f} ± {std_auc:.4f}")


    # ================================
    # TITLE PAGE
    # ================================
    plt.figure(figsize=(11,8.5)); plt.axis('off')
    title = "ML Hybrid Models Comparison Report"
    meta = f"""Generated: {datetime.now().isoformat()}
Dataset samples: {len(y)}
Models: {', '.join(models_final)}
StratifiedKFold: {cfg.N_SPLITS} splits
Random seed: {cfg.RANDOM_STATE}"""

    plt.text(0.5, 0.6, title, ha='center', va='center', fontsize=26, weight='bold')
    plt.text(0.5, 0.45, meta, ha='center', va='center', fontsize=10)
    plt.tight_layout()
    pdf_pages.savefig()
    plt.close()


    # ================================
    # COMPARISON TABLE
    # ================================
    df = pd.DataFrame(results_summary).sort_values(by="mean_roc_auc", ascending=False).reset_index(drop=True)
    df.to_csv(COMPARISON_CSV, index=False)

    print("\n=== Comparison Table ===")
    print(df)

    plt.figure(figsize=(11,8.5)); plt.axis('off')
    plt.text(0.02,0.98,"Model Comparison Table (sorted by mean ROC AUC)", fontsize=14, weight='bold', va='top')
    plt.text(0.02,0.92, df.to_string(index=False), fontsize=8, family='monospace', va='top')
    pdf_pages.savefig()
    plt.close()

print("\nPDF saved to:", pdf_path)
print("CSV saved to:", COMPARISON_CSV)



===== CLUSTERS: 5 =====
✔ Dataset OK (2018, 447)
GBDT-SVM: 0.9117 ± 0.0194
XGB: 0.9405 ± 0.0120
[LightGBM] [Info] Number of positive: 808, number of negative: 806
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003774 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 85113
[LightGBM] [Info] Number of data points in the train set: 1614, number of used features: 446
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500620 -> initscore=0.002478
[LightGBM] [Info] Start training from score 0.002478


c:\Users\naim1\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 808, number of negative: 806
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003438 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 85239
[LightGBM] [Info] Number of data points in the train set: 1614, number of used features: 446
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500620 -> initscore=0.002478
[LightGBM] [Info] Start training from score 0.002478


c:\Users\naim1\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 808, number of negative: 806
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004132 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 85042
[LightGBM] [Info] Number of data points in the train set: 1614, number of used features: 446
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500620 -> initscore=0.002478
[LightGBM] [Info] Start training from score 0.002478


...

[LightGBM] [Info] Number of positive: 808, number of negative: 799
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002859 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 84939
[LightGBM] [Info] Number of data points in the train set: 1607, number of used features: 446
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.502800 -> initscore=0.011201
[LightGBM] [Info] Start training from score 0.011201


c:\Users\naim1\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 808, number of negative: 799
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003187 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 84994
[LightGBM] [Info] Number of data points in the train set: 1607, number of used features: 446
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.502800 -> initscore=0.011201
[LightGBM] [Info] Start training from score 0.011201


c:\Users\naim1\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


ENSEMBLE: 0.9359 ± 0.0093

=== Comparison Table ===
     clusters     model  mean_roc_auc       std
0          20  ENSEMBLE      0.946206  0.010673
1          20       LGB      0.945616  0.010153
2          20       XGB      0.944547  0.011293
3          10       LGB      0.943382  0.006973
4          10  ENSEMBLE      0.942718  0.006617
..        ...       ...           ...       ...
121        11  GBDT-SVM      0.900527  0.004706
122        23  GBDT-SVM      0.898086  0.010139
123        17  GBDT-SVM      0.897074  0.015689
124        12  GBDT-SVM      0.895815  0.010271
125         6  GBDT-SVM      0.894872  0.012573

[126 rows x 4 columns]

PDF saved to: results/ML_Report.pdf
CSV saved to: results/comparison.csv
